### 2-1-1. 출력 예시
이 실습 이제 아래에서 할거임
```
[11:49:01] 김사과: 오늘 회의를 시작하겠습니다.
[11:49:11] 오렌지: 네, 첫 번째 안건을 말씀드리겠습니다.
[11:50:03] 김사과: 예산안을 다시 검토해보세요.
```

### **2-2. 화자 분리와 화자 식별**
- 화자 분리: 녹음에서 '언제 누가 말했는지'를 시간 구간으로 나누는 작업
- 컴퓨터는 말투, 발성, 음색을 신경망으로 고정 길이 숫자 벡터(화자 임베딩)로 바꿈 > 문장 뜻보다 누구인지 가르는 특징을 담음
    - 오디오를 읽고 16Hz 모노로 맞춤
    - 짧은 창마다 발화인지, 여러 사람이 겹치는지 봄(무음과 발화의 경계가 중요 > 이걸 음성 활동 탐지 VAD라 함)
    - 발화 구간마다 임베딩을 만듦
    - 비슷한 임베딩을 묶어서 같은 화자로 보게 함
    - 묶음 결과를 시간축에 다시 올려 SPEAKER_00 같은 라벨을 붙여줌
- 일반 화자 분리는 두 사람이 동시에 말한 겹침을 그대로 둘 수 있음. Exclusive 화자 분리는 한 시각에 화자를 한 명만 남김(ASR에 넘기기 쉽지만, 실제 겹침이 사라진 것 처럼 보임)
- 화자 식별: '이 목소리가 등록된 누구인가?' 라는 참가자 음성 임베딩을 저장하고 비교함

In [16]:
# 여기에 import 필요한거 모아모아했음
import os
import random
import numpy as np
import sounddevice as sd
import torch
import threading
import queue
from typing import Literal, TypedDict
from datetime import datetime, timedelta
from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from qwen_asr import Qwen3ASRModel
from pyannote.audio import Inference, Model, Pipeline

In [2]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [3]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN Error!!")

In [4]:
# 설정할 값 상수 정의

SR = 16000 # 16KHz 모노 1초 = 16,000 샘플
MIN_RMS = 0.00001 # 이 값보다 RMS가 작으면 말이 없는 구간으로 보고 전사하지 않음(내 노트북은 이렇게 설정해야 인식됨)
PAUSE_SECONDS = 0.6 # 말이 이 시간만큼 끊기면 한 구간으로 넘김. 더 길어지면 말 중간이라도 넘길 수 있음
MAX_CHUNK = 12.0
MATCH_THRESHOLD = 0.60 # 정규화 임베딩의 내적(코사인 유사도)이 이 값보다 늦으면 등록된 사람으로 보지 않음

In [5]:
embedding_model = Model.from_pretrained("pyannote/embedding", token=HF_TOKEN)
embedding_model.to(DEVICE)
# window="whole" : 음성 전체를 넘기라는 뜻
embedder = Inference(embedding_model, window="whole")

d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:197: Redirecting import of pytorch_lightning.callbacks.early_stopping.EarlyStopping to lightning.pytorch.callbacks.early_stopping.EarlyStopping
d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:197: Redirecting import of pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint to lightning.pytorch.callbacks.model_checkpoint.ModelCheckpoint
d:\OSA\14_Agent\qwen_asr_test\.venv\Lib\site-packages\lightning\pytorch\utilities\migration\migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
INFO: Lightning automatically upgraded your loaded checkp

In [6]:
# 누가 말했는지 화자 분리 파이프라인 만들거임
diarization = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
diarization.to(DEVICE)

# bfloat16 지원하지 않은 디바이스면 float32 쓰면 됨
ASR_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
asr_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if torch.cuda.is_available() else torch.float32

asr_model = Qwen3ASRModel.from_pretrained(
    ASR_MODEL_ID, dtype=asr_dtype, device_map="cuda" if torch.cuda.is_available() else "cpu", max_inference_batch_size=1, max_new_tokens=256
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.53s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# **2-3. 목소리 등록과 이름 찾기**

![아래 함수 설명](./images/ref/07_03.png)

In [7]:
# 음성을 녹음하는 함수
def record_voice(seconds=12, prompt=True):
    # prompt가 True라면 말을 할 수 있게
    if prompt:
        print(f"{seconds}초 녹음 중... 자연스럽게 말하세요.")
    # 1초가 SR이니까 그만큼 곱해서 쓰는거임
    recording = sd.rec(int(SR * seconds), samplerate=SR, channels=1, dtype='float32')
    sd.wait()
    return recording[:, 0].copy()

# 음성에서 화자 임베딩을 추출하는 함수
def embedding_of(wave):
    wave = np.asanyarray(wave, dtype=np.float32).reshape(-1)
    # 1초 미만이면 진행하지 않도록
    if len(wave) < SR:
        raise ValueError("화자 식별에는 최소 1초 이상의 음성이 필요합니다.")
    tensor = torch.from_numpy(wave).unsqueeze(0)  
    with torch.inference_mode():
        vector = np.asarray(
            embedder({"waveform": tensor, "sample_rate": SR})
        ).reshape(-1)
    norm = np.linalg.norm(vector)
    # 벡터의 노름이 너무 작으면 유효하지 않은 특징으로 간주
    # norm: 벡터의 크기(노름)
    # 1e-10보다 작으면 거의 0에 가까운 벡터로 간주
    # 이 경우 유효한 화자 특징을 추출하지 못한 것으로 판단
    if norm < 1e-10:
        raise ValueError("유효한 화자 특징을 추출하지 못했습니다.")
    return vector / norm

# 등록된 화자 정보를 저장하는 딕셔너리
registered = {}

# 화자를 등록하는 함수
def enroll(name, seconds=12):
    name = name.strip()  # 이름의 앞뒤 공백 제거
    if not name:
        raise ValueError("이름을 입력해야 합니다.")
    # 녹음하게 함
    audio = record_voice(seconds)
    # 녹음된 음성의 RMS(Root Mean Square) 값으로, 음성의 세기를 나타냄
    rms = float(np.sqrt(np.mean(audio.astype(np.float64)**2)))
    # 위에서 상수로 설정한값보다 작으면 음성이 너무 작다고 판단
    if rms < MIN_RMS:
        raise ValueError("녹음된 음성이 너무 작습니다. 다시 시도하세요.")
    # 그게 아니라면 화자 임베딩을 추출하여 등록
    registered[name] = embedding_of(audio)
    print(f"등록 완료: {name} (RMS={rms:.4f})")

# 입력 벡터와 등록된 화자 벡터를 비교하여 가장 유사한 화자를 찾는 함수
def match_speaker(vector):
    if not registered:
        raise ValueError("등록된 화자가 없습니다.")
    # dot: 이 함수는 내적 구하는 함수임
    # 각 등록된 화자와 입력 벡터 간의 유사도를 계산
    # why? 내적을 통해 두 벡터 간의 유사도를 측정하기 위해서임. 내적 값이 클수록 두 벡터가 유사함을 의미.
    scores = {name: float(np.dot(reference, vector)) for name, reference in registered.items()}
    # best와 그 점수를 반환
    best = max(scores, key=scores.get)
    if scores[best] < MATCH_THRESHOLD:
        return "미등록"
    return best

In [8]:
enroll("김사과")

12초 녹음 중... 자연스럽게 말하세요.
등록 완료: 김사과 (RMS=0.0000)


In [9]:
enroll("오렌지")

12초 녹음 중... 자연스럽게 말하세요.
등록 완료: 오렌지 (RMS=0.0004)


In [10]:
print("현재 등록된 참석자:", list(registered))

현재 등록된 참석자: ['김사과', '오렌지']


### **2-4. 회의 Supervisor**
- 마이크 스레드는 그래프와 따로 돌게 만들거임
- 감독자(랭그래프의 Supervisor)는 큐에서 받은 구간을 보고 다음 담당자를 고름
- 담당자는 자기 필드만 채운 뒤 감독자에게 돌아옴

```
담당자
capture: 말이 끊겨서 큐에 들어온 구간을 꺼냄 (말이 끊겻다는건 12초가 되었다는거) + 마이크는 멈추지 않음
diarize: 화자 분리로 발화 시각을 나누는 에이전트
identify: 화자별 임베딩을 등록 음성과 비교. 임계값 미만은 미등록
asr: 현재 발화를 한국어 텍스트로 바꿈
report: "[시:분:초] 이름: 문장"을 출력하고 다음 발화로 넘어감
```

우리는 큐로 데이터 처리할거임
- 큐: FIFO(First In First Out) 구조 = 먼저 들어온 데이터부터
- 스택: LIFO(Last In First Out) 구조 = 마지막에 들어온 데이터부터

**[마이크는 음성 데이터를 계속 생산(Producer)하고, 큐는 데이터를 순서대로 보관하며, 분석 작업은 데이터를 꺼내 소비(Consumer)하는 구조]**

왜 큐를 선택?
> 회의 음성은 발화 순서가 중요하니까!!

- thred: 작업 단위임. 멀티스레딩을 통해 여러 스레드가 동시에 실행될 수 있음.
- threading: 여러 작업을 동시에 수행할 수 있게 해주는 파이썬 모듈. 멀티스레딩을 통해 병렬 처리 가능.

왜 멀티스레딩 사용해?
> 음성 인식 처리하는동안 마이크 녹음이 중단되면 그 사이에 다른 사람이 말한 내용을 놓칠 수 있으니까!

![멀티스레딩 사용 이유](./images/ref/07_04.png)

그럼 self._lock = threading.Lock() 이건 뭔데?
> 여러 스레드가 동일한 데이터를 동시에 수정하면서 발생할 수 있는 데이터 충돌 방지용!

![락 거는 이유](./images/ref/07_05.png)

- stream: 데이터를 한 번에 전부 가져오는거 아니고! 연속적으로 전달받을 수 있도록 하는거임.

- 버퍼: 메모리의 일부.. 버퍼링.. 데이터가 너무 크고 예측이 안되더라도 일시적으로 저장해두는 공간

In [ ]:
# 마이크 입력을 통해 실시간으로 음성을 수집하고 처리하는 클래스
class MeetingMic:
    # 초기화 메서드: 클래스가 생성될 때 호출되어 필요한 속성을 초기화
    def __init__(self):
        self._parts = []                    # 수집된 음성 블록 저장
        self._samples = 0                   # 현재까지 수집된 전체 샘플 개수
        self._silent_samples = 0            # 연속적으로 감지된 무음 샘플 개수
        self._heard = False                 # 현재 구간에서 발화로 간주되는 소리가 감지되었는지 여부(소리 들렸는지?)
        self._lock = threading.Lock()       # 여러 스레드(여러 스레드 = 동시에 실행되는 작업)의 동시 접근 방지
        self._ready = queue.Queue()         # 처리가 완료된 음성 구간 저장
        self._stream = sd.InputStream(      # 마이크 입력 스트림 객체
            samplerate=SR,
            channels=1,  
            blocksize=int(SR * 0.1), 
            callback=self._on_audio         # ⭐ 음성 블록이 준비될 때 실행할 함수
        )

    # 음성 버퍼 및 상태 초기화
    def _reset(self, keep_tail=False):
        if keep_tail and self._parts:
            # 전체적인 데이터에서 앞부분은 제외하고 끝에 살짝 0.3초만 남겨서 다음 구간과 이어지도록 함
            # why? 사람이 말하기를 시작하거나 끝날때 흐름이 끊기지 않도록 하기 위함(말 끝을 흐리는 경우...)
            # 버퍼를 쪼개서 쓸 때 많이 사용하는 방법임(이런 방법도 있다 알기)
            tail = np.concatenate(self._parts)[-int(0.3 * SR):]
            self._parts = [tail] if len(tail) else []
        else:
            # 현재까지 수집된 음성과 관련 상태를 모두 초기화
            # 주로 하나의 음성 구간이 완성되어 Queue에 전달할 때 사용
            self._parts = []
            self._samples = 0
        self._silent_samples = 0
        self._heard = False

    # ⭐ 실시간 음성 수집 + 발화 구간 분리 -> 초기화 메서드에 callback으로 등록되어있어 자동 실행될거임
    # indata: 마이크에서 수집된 실제 음성 데이터
    # frames: 현재된 오디오 프레임 개수
    # time_info: 오디어 입력과 callback 실행에 관련된 시간 정보
    # status: 입력 오버플로등의 스트림 상태 정보(오버플로 = 입력이 내가 원하는 사이즈보다 넘치는..경우)
    # 이 매개변수들은 내가 채우는게 아니고 sounddevice가 자동으로 채워서 전달해줌(직접 전달 XX!!)
    def _on_audio(self, indata, frames, time_info, status):
        block = indata[:, 0].copy()
        rms = float(np.sqrt(np.mean(block.astype(np.float64) ** 2)))
        item = None
        with self._lock:
            self._parts.append(block)
            self._samples += len(block)
            # rms가 기준값보다 작으면 무음 샘플 누적
            # 기준값 이상이면 연속 무음 기록 초기화 + 발화 감지
            # > 사람이 말하다 멈추면 무음 샘플 쌓이고, 다시 말하면 무음 샘플은 0으로 들어감
            if rms < MIN_RMS:
                self._silent_samples += len(block)
            else:
                self._silent_samples = 0
                self._heard = True
            # 발화 없이 계속 무음이라면!?
            # dur = 현재까지 수집한 음성의 길이(초)
            dur = self._samples / SR
            if not self._heard:
                # 아무도 말하지 않은 상태로 2초를 초과하면 _reset 호출해서 마지막 0.3초 남기는거 진행
                # why? 불필요한 무음 데이터 쌓이는거 방지하고, 새로운 발화 시작될 때 경계 음성 손실을 줄이려고
                if dur > 2.0:
                    self._reset(keep_tail=True)
                return
            # 이제 음성을 자를 시점을 결정하는데 이거 참 중요하다!!!!
            # 1. cut_for_pause: 사람이 말하다 멈춘 경우(연속 무음)
            # 2. cut_for_len: 음성 길이가 최대치에 도달한 경우
            # 둘 다 충족하지 않으면 return해서 이번 콜백은 종료하고, 다음 음성 블록 기다리면 됨
            cut_for_pause = dur >= 1.0 and self._silent_samples / SR >= PAUSE_SECONDS
            cut_for_len = dur >= MAX_CHUNK
            if not (cut_for_pause or cut_for_len):
                return
            # 이제 0.1초 단위로 저장한 음성 블록을 배열로 합치는데
            audio = np.concatenate(self._parts)
            # 현재 시간에서 녹음된 음성의 길이를 빼서 해당 음성 구간의 시작 계산을 계산하는 코드
            # 그니까 회의록에 시간 정보를 기록할 때 활용되는거임!
            # 이게 나중에 시간을 구하면 실제 녹음 시작 시간과는 약간 차이가 있을 수 있음
            # 그래서 좀 오래 걸리더라도 여기서 정확히 시작 시간을 계산해두는게 좋음 (강사님 말씀~~)
            started = datetime.now() - timedelta(seconds=len(audio) / SR)
            # 끝에 불필요한 무음 제거하기 (마지막 연속 무음 제거)
            # 앞에서 설명한 그 음성 자르는거에서 연속 무음이 끝나는 지점을 기준으로 잘라주는 부분임
            # 근데 자연스럽게 음성 경계 설정하려고 0.15초 정도 여유두는거임
            # >> RMS 기반 무음 판단은 완벽하지 않아서 발화 마지막 음절이나 작은 소리 잘릴 수 있다 Oh no!
            if cut_for_pause and not cut_for_len:
                keep = max(int(0.3 * SR), len(audio) - self._silent_samples + int(0.15 * SR))
                audio = audio[:keep]
            # 현재 음성 구간 수집 끝났으니까 버퍼 초기화하고
            self._reset()
            # 완성된 음성 데이터랑 시작 시간을 튜블로 만들어서 큐에 넣음!
            item = (audio, started)
        # 데이터(item)을 계속 쌓아줌
        self._ready.put(item)

    # 음성 수집 시작
    def start(self):
        # lock가 True 라면? 버퍼 초기화하고
        with self._lock:
            self._reset()
        while True:
            try:
                # 기존 큐 비운다음
                self._ready.get_nowait()
            except queue.Empty:
                break
        # 마이크 시작
        if not self._stream.active:
            self._stream.start()

    # 음성 수집 중지 -> 활성화 되어있을때만
    def stop(self):
        if self._stream.active:
            self._stream.stop()

    # 스트림 종료 + 자원 해제(스트림 재생성 필요함)
    def close(self):
        self.stop()
        self._stream.close()

    # 음성 데이터 가져오기
    def take(self):
        return self._ready.get()
        
# Jupyter Notebook에서 셀을 재실행할 경우를 대비한 코드
# 기존 마이크 객체가 존재하면 스트림을 종료하고 시스템 자원을 해제
# why? 마이크 자원 충돌 및 중복 실행 방지
if "meeting_mic" in globals():
    meeting_mic.close()

meeting_mic = MeetingMic()